In [1]:
import sys
import os

# Add project root to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from utils.remove_correlated_features import remove_correlated_features
from utils.get_data_for_feature_selection import get_player_data_by_pos
from utils.perform_xgboost_selection import perform_xgboost_selection

# Feature Selection 

The aim is to create a model for prodicting points for players in fpl. Since each position gets points based on different parameters, it is decided that one regression model is to be implemented for each position (GK, DEF, MID, FWD).

The goal of this analysis is to decide on what features are relevant for each position, this in order to reduce the compute needed to create models (I run everything locally). 

Initial plan of using correlation metrict to perform feature selection is to fragile. Using XGBoost instead based on this article; 
https://medium.com/@dhanyahari07/feature-selection-using-xgboost-f0622fb70c4d

Training and feature selection is performed on data from 22/23 and 23/24 season. The final evaluation is performed on data from the 24/25 season.


##### On using XGBoost as a feature selection pipeline
XGBoost calculates three types of feature importance scores:

* Gain: Average loss reduction gained when using a feature for splitting.

* Cover: The number of times a feature is used to split data across trees weighted by training data points.

* Weight: Total number of times a feature is used to split data across all trees.

#### Feature Selection Hyperparameters

In [2]:
TARGET = 'total_points'  
VARIANCE_TRESHOLD: float = 0.015     # Minimum variance for a feature to be kept
CORR_FEATURE_TRESHOLD: float = 0.8  # Maximum correlation allowed between two features
PRINT_INFO: bool = True

#### Relevant position featurs

In [3]:
FIELD_PLAYER_RELEVANT_FEATURES = [
                'name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets', 
                'creativity', 'expected_assists', 'expected_goal_involvements', 
                'expected_goals','expected_goals_conceded', 'goals_scored', 'ict_index', 'influence', 'minutes', 
                'opponent_team', 'own_goals', 'penalties_missed', 'red_cards', 
                'selected', 'team_a_score', 'team_h_score', 'threat', 'total_points', 
                'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 
                'yellow_cards', 'GW'
                ]

GOALKEEPER_RELEVANT_FEATURES = [
                'name', 'position', 'GW', 'xP', 'bonus', 'bps', 'clean_sheets',  
                'expected_goals_conceded', 'ict_index', 'influence', 'minutes', 
                'opponent_team', 'own_goals', 'red_cards', 'saves',
                'selected', 'team_a_score', 'team_h_score', 'total_points', 
                'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 
                'yellow_cards', 
                ]

### Getting data and performing selection

#### FWD

In [4]:
# Get data for FWD players

fwd_data = get_player_data_by_pos('FWD', FIELD_PLAYER_RELEVANT_FEATURES)

fwd_data = remove_correlated_features(fwd_data, VARIANCE_TRESHOLD, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = fwd_data['total_points']  # Extract the target column
fwd_data = fwd_data.drop(columns=['total_points'])  # Drop the target column from the features

fwd_features = perform_xgboost_selection(fwd_data, target_col)

print("\nFinal selected features for FWD:")
for feature in fwd_features:
    print(feature)

# Store the selected features to a CSV file
pd.DataFrame(fwd_features).to_csv("../data/features/fwd_features.csv", index=False, header=False)

# clear memory
del fwd_data, fwd_features, target_col

number of datapoints before filtering: 6967
number of datapoints after filtering: 3309
Removed 61 low-variance features.
Found 0 pairs of highly correlated features.
Removed 0 features due to high correlation.
Original number of features; 140. Final number of features: 79
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 78 different feature thresholds...


c:\Users\trygt\fpl_dir\fpl_team_selector\.venv\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Thresh=0.0038, n=78, MSE=10.2943, Best Score=10.2943


c:\Users\trygt\fpl_dir\fpl_team_selector\.venv\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Thresh=0.0052, n=77, MSE=10.1238, Best Score=10.1238


...

Thresh=0.0301, n=3, MSE=10.7760, Best Score=9.9622


c:\Users\trygt\fpl_dir\fpl_team_selector\.venv\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Thresh=0.0307, n=2, MSE=11.7098, Best Score=9.9622


c:\Users\trygt\fpl_dir\fpl_team_selector\.venv\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Thresh=0.0310, n=1, MSE=10.2804, Best Score=9.9622

Optimal number of features found: 52 (with MSE: 9.9622)

Feature Selection Complete

Final selected features for FWD:
strength_difference
minutes_lag1
minutes_lag4
minutes_lag5
team_a_score_lag3
transfers_balance_lag1
transfers_balance_lag3
transfers_balance_lag4
value_lag1
value_lag2
value_lag4
value_lag5
team_h_score_lag2
team_h_score_lag5
total_points_lag1
total_points_lag2
total_points_lag4
total_points_lag5
transfers_out_lag2
transfers_out_lag3
transfers_out_lag4
xP_lag1
xP_lag3
xP_lag5
threat_lag3
threat_lag4
threat_lag5
transfers_in_lag1
transfers_in_lag2
transfers_in_lag3
transfers_in_lag4
creativity_lag1
creativity_lag2
creativity_lag3
creativity_lag5
ict_index_lag1
ict_index_lag3
ict_index_lag4
ict_index_lag5
influence_lag1
influence_lag2
influence_lag4
influence_lag5
bps_lag1
bps_lag2
bps_lag3
bps_lag5
selected_lag1
selected_lag2
selected_lag3
selected_lag4
selected_lag5


#### MID

In [8]:
mid_data = get_player_data_by_pos('MID', FIELD_PLAYER_RELEVANT_FEATURES)
mid_features = perform_feature_selection(mid_data,
                                         TARGET,
                                         CORR_TARGET_TRESHOLD, 
                                         VARIANCE_TRESHOLD, 
                                         CORR_FEATURE_TRESHOLD, 
                                         PRINT_INFO)

for feature in mid_features:
    print(feature)

# Store the selected features for FWD to a CSV file
mid_features_df = pd.DataFrame(mid_features)

if 'total_points' in mid_features_df.columns:
    mid_features_df.drop('total_points', axis=1, inplace=True)  # Remove target variable from features

for feature in mid_features_df.columns:
    print(f"{feature}")
mid_features_df.to_csv("../data/features/mid_features.csv", index=False, header=False)

# clear memory
del mid_data, mid_features, mid_features_df

number of datapoints before filtering: 24265
number of datapoints after filtering: 11989
Removed 15 low-variance features.
Found 60 pairs of highly correlated features.
Removed 34 features due to high correlation.
Original number of features; 140. Final number of features: 91
total_points
was_home
defence_strenght_difference
strength_difference
clean_sheets_lag1
clean_sheets_lag2
clean_sheets_lag3
clean_sheets_lag4
clean_sheets_lag5
expected_goal_involvements_lag1
expected_goal_involvements_lag2
expected_goal_involvements_lag3
expected_goal_involvements_lag4
expected_goal_involvements_lag5
creativity_lag1
creativity_lag2
creativity_lag3
creativity_lag4
creativity_lag5
bonus_lag1
bonus_lag2
bonus_lag3
bonus_lag4
bonus_lag5
value_lag1
xP_lag1
xP_lag2
xP_lag3
xP_lag4
xP_lag5
expected_assists_lag1
expected_assists_lag2
expected_assists_lag3
expected_assists_lag4
expected_assists_lag5
selected_lag1
team_h_score_lag1
team_h_score_lag2
team_h_score_lag3
team_h_score_lag4
team_h_score_lag5
yel

#### DEF

In [9]:
def_data = get_player_data_by_pos('DEF', FIELD_PLAYER_RELEVANT_FEATURES)
def_features = perform_feature_selection(def_data,
                                         TARGET,
                                         CORR_TARGET_TRESHOLD, 
                                         VARIANCE_TRESHOLD, 
                                         CORR_FEATURE_TRESHOLD, 
                                         PRINT_INFO)

for feature in def_features:
    print(feature)

# Store the selected features for FWD to a CSV file
def_features_df = pd.DataFrame(def_features)

if 'total_points' in def_features_df.columns:
    def_features_df.drop('total_points', axis=1, inplace=True)  # Remove target variable from features

for feature in def_features_df.columns:
    print(f"{feature}")
def_features_df.to_csv("../data/features/def_features.csv", index=False, header=False)

# clear memory
del def_data, def_features, def_features_df

number of datapoints before filtering: 18794
number of datapoints after filtering: 8836
Removed 25 low-variance features.
Found 32 pairs of highly correlated features.
Removed 20 features due to high correlation.
Original number of features; 140. Final number of features: 95
total_points
was_home
attack_strenght_difference
strength_difference
clean_sheets_lag1
clean_sheets_lag2
clean_sheets_lag3
clean_sheets_lag4
clean_sheets_lag5
expected_goal_involvements_lag1
expected_goal_involvements_lag2
expected_goal_involvements_lag3
expected_goal_involvements_lag4
expected_goal_involvements_lag5
creativity_lag1
creativity_lag2
creativity_lag3
creativity_lag4
creativity_lag5
bonus_lag1
bonus_lag2
bonus_lag3
bonus_lag4
bonus_lag5
value_lag2
xP_lag1
xP_lag2
xP_lag3
xP_lag4
xP_lag5
goals_scored_lag1
goals_scored_lag2
goals_scored_lag3
goals_scored_lag4
goals_scored_lag5
selected_lag1
threat_lag1
threat_lag2
threat_lag3
threat_lag4
threat_lag5
team_h_score_lag1
team_h_score_lag2
team_h_score_lag3
t

#### GK

In [10]:
gk_data = get_player_data_by_pos('GK', FIELD_PLAYER_RELEVANT_FEATURES)
gk_features = perform_feature_selection(gk_data,
                                         TARGET,
                                         CORR_TARGET_TRESHOLD, 
                                         VARIANCE_TRESHOLD, 
                                         CORR_FEATURE_TRESHOLD, 
                                         PRINT_INFO)

for feature in gk_features:
    print(feature)

# Store the selected features for FWD to a CSV file
gk_features_df = pd.DataFrame(gk_features)

if 'total_points' in gk_features_df.columns:
    gk_features_df.drop('total_points', axis=1, inplace=True)  # Remove target variable from features

for feature in gk_features_df.columns:
    print(f"{feature}")
gk_features_df.to_csv("../data/features/gk_features.csv", index=False, header=False)

# clear memory
del gk_data, gk_features, gk_features_df

number of datapoints before filtering: 6204
number of datapoints after filtering: 1649
Removed 40 low-variance features.
Found 36 pairs of highly correlated features.
Removed 23 features due to high correlation.
Original number of features; 140. Final number of features: 77
total_points
was_home
attack_strenght_difference
strength_difference
clean_sheets_lag2
influence_lag2
influence_lag3
influence_lag5
creativity_lag1
creativity_lag2
creativity_lag3
creativity_lag4
creativity_lag5
bonus_lag1
bonus_lag2
bonus_lag3
bonus_lag4
bonus_lag5
value_lag4
xP_lag1
xP_lag2
xP_lag3
xP_lag4
xP_lag5
selected_lag4
threat_lag1
threat_lag2
threat_lag3
threat_lag4
threat_lag5
team_h_score_lag1
team_h_score_lag2
team_h_score_lag3
team_h_score_lag4
team_h_score_lag5
yellow_cards_lag1
yellow_cards_lag2
yellow_cards_lag3
yellow_cards_lag4
yellow_cards_lag5
transfers_out_lag1
transfers_out_lag2
transfers_out_lag3
transfers_out_lag4
transfers_out_lag5
transfers_in_lag1
transfers_in_lag2
transfers_in_lag3
tran